# Knowledge Distillation

- distillation using forward KL divergence on token distributions (logits)
- teacher outputs are truncated with top-p-k (p>=.95, k<=100) before forming the target distribution to save storage space
- KD loss weight λ follows a **warmup → stable → decay** schedule (warmup-stable-decay by Hu et al)
- learning rate follows WSD pattern to promote maximum learning in stable period w/ teacher model and decay towards student at end (Peng et al)

In [ ]:
import os
import math
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import GPT2LMHeadModel

from model import GPT, GPTConfig

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype       = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
print(f"device: {device}  |  dtype: {dtype}")

# ── Data ──────────────────────────────────────────────────────────────────────
block_size = 1024
batch_size = 3

fineweb_shards = sorted([
    os.path.join('edu_fineweb10B', f) for f in os.listdir('edu_fineweb10B')
    if f.startswith('edufineweb_train') and f.endswith('.npy')
])
commonpile_shards = sorted([
    os.path.join('common_pile_filtered', f) for f in os.listdir('common_pile_filtered')
    if f.startswith('commonpile_train') and f.endswith('.npy')
]) if os.path.exists('common_pile_filtered') else []

train_shards = fineweb_shards * 3 + commonpile_shards
random.shuffle(train_shards)

val_data   = np.load(os.path.join('edu_fineweb10B', 'edufineweb_val_000000.npy'))
train_data = np.load(train_shards[0])

def get_batch(split): # same logic from train.py
    data = train_data if split == 'train' else val_data
    ix   = torch.randint(len(data) - block_size, (batch_size,))
    x    = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64))     for i in ix])
    y    = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

device: cuda  |  dtype: torch.bfloat16


## Helper Functions

- filtering for p and k of teacher logits
- computing WSD KD divergence weighting
- computing learning rate based on WSD
- precompute for teacher for offline distillation (don't need student and teacher running on GPU at once)

In [8]:
def top_p_k_filter(logits: torch.Tensor, top_k: int = 100, top_p: float = 0.95) -> torch.Tensor:
    """
    Apply top-p (probability) then top-k (highest ranked) filtering to logits from teacher
    Tokens outside the kept set are set to -inf so softmax gives them zero probability.
    """
    # 1. Top-p
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    # A token is removed when ALL tokens ranked above it already cover >= top_p mass.
    remove = (cumulative - sorted_probs) >= top_p
    remove_mask = torch.zeros_like(logits, dtype=torch.bool).scatter_(-1, sorted_idx, remove)
    logits = logits.masked_fill(remove_mask, float('-inf'))

    # 2. Top-k for survivors
    k = min(top_k, logits.size(-1))
    kth_val = torch.topk(logits, k, dim=-1).values[..., -1, None]
    return logits.masked_fill(logits < kth_val, float('-inf'))


def kd_lambda(step: int, warmup: int, stable: int, decay: int) -> float:
    """
    Returns scalar for warmup-stable-decay (WSD)
    Used for KD loss weight and learning rate
    1. warmup ramps linearly from 0-1
    2. stable regime stays at 1 (teacher-only contribution to loss)
    3. cosine decay from 1 to 0
    """
    if step < warmup:
        return step / warmup
    if step < warmup + stable:
        return 1.0
    progress = min(1.0, (step - warmup - stable) / decay)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def wsd_lr(lam:float, max_lr: float, min_lr: float) -> float:
    """
    WSD-LR: warmup-stable-decay learning rate schedule.
    Uses the same phase boundaries as WSD lam but mapped to min/max lr
    """
    # map from 0-1 to min_lr to max_lr
    return min_lr + (max_lr-min_lr) * lam

def kd_loss_from_sparse(
    student_logits: torch.Tensor,
    top_idx: torch.Tensor,
    top_probs: torch.Tensor,
) -> torch.Tensor:
    """
    Forward KL divergence from pre-computed sparse teacher outputs.
    Avoids reconstructing the full dense teacher distribution.

    Args:
        student_logits: [B, T, 50304]
        top_idx:        [B, T, k]  int64  — vocab indices of kept tokens
        top_probs:      [B, T, k]  float  — renormalized teacher probs at those indices
    """
    s_log      = F.log_softmax(student_logits[..., :TEACHER_V], dim=-1)  # [B, T, V]
    s_log_at_k = s_log.gather(-1, top_idx)                               # [B, T, k]
    t_log_at_k = top_probs.clamp(min=1e-10).log()
    kl = (top_probs * (t_log_at_k - s_log_at_k)).sum(-1)                 # [B, T]
    return kl.mean()


def precompute_teacher_sparse(shard_path, teacher, top_k, top_p, block_size, batch_size):
    """
    Run the teacher on every non-overlapping block_size chunk of a shard and
    save the sparse (top_idx, top_probs) outputs as companion .npy files.

    Companion files written next to each shard:
      <shard>_tidx.npy   uint16  [num_chunks, block_size, top_k]  vocab indices
      <shard>_tprob.npy  float16 [num_chunks, block_size, top_k]  renorm'd probs

    uint16 covers 0-65535, sufficient for GPT-2 vocab (50257).
    Idempotent: skips shard if both companion files already exist.
    """
    tidx_path  = shard_path.replace('.npy', '_tidx.npy')
    tprob_path = shard_path.replace('.npy', '_tprob.npy')
    if os.path.exists(tidx_path) and os.path.exists(tprob_path):
        print(f"  skip (exists): {os.path.basename(shard_path)}")
        return

    data       = np.load(shard_path)
    num_chunks = len(data) // block_size
    all_idx    = np.empty((num_chunks, block_size, top_k), dtype=np.uint16)
    all_prob   = np.empty((num_chunks, block_size, top_k), dtype=np.float16)

    for b_start in range(0, num_chunks, batch_size):
        b_end  = min(b_start + batch_size, num_chunks)
        chunks = np.stack([data[j*block_size:(j+1)*block_size] for j in range(b_start, b_end)])
        x      = torch.from_numpy(chunks.astype(np.int64)).to(device)

        with torch.no_grad(), torch.amp.autocast(device_type=device_type, dtype=dtype):
            logits = teacher(input_ids=x).logits.float()        # [b, T, 50257]

        probs  = F.softmax(top_p_k_filter(logits, top_k, top_p), dim=-1)
        tv, ti = torch.topk(probs, top_k, dim=-1)
        all_idx [b_start:b_end] = ti.cpu().numpy().astype(np.uint16)
        all_prob[b_start:b_end] = tv.cpu().numpy().astype(np.float16)

        if b_start % (batch_size * 50) == 0:
            print(f"  {os.path.basename(shard_path)}: {b_end}/{num_chunks} chunks", end='\r')

    np.save(tidx_path,  all_idx)
    np.save(tprob_path, all_prob)
    print(f"\n  saved: {os.path.basename(tidx_path)}, {os.path.basename(tprob_path)}")


def get_train_batch(data, tidx, tprob):
    """
    Aligned-chunk sampling from whichever shard is passed in.
    Chunk index i maps directly to tidx[i] / tprob[i], keeping tokens and
    sparse teacher distribution in sync across shards.
    Returns x, y (tokens) and ti, tp (sparse teacher distribution).
    """
    num_chunks = len(data) // block_size
    cids = torch.randint(num_chunks, (batch_size,)).tolist()
    x  = np.stack([data[i*block_size  :(i+1)*block_size  ] for i in cids]).astype(np.int64)
    y  = np.stack([data[i*block_size+1:(i+1)*block_size+1] for i in cids]).astype(np.int64)
    ti = np.stack([tidx [i] for i in cids]).astype(np.int64)
    tp = np.stack([tprob[i] for i in cids]).astype(np.float32)
    return (torch.from_numpy(x).to(device),  torch.from_numpy(y).to(device),
            torch.from_numpy(ti).to(device), torch.from_numpy(tp).to(device))

## Training param setup

In [9]:
# ── Schedule (10% warmup, 89% stable, 1% decay) ──────────────────────
max_iters    = 5000
warmup_iters = int(0.10 * max_iters)   # 10% warmup
stable_iters = int(0.89 * max_iters)   # 89% stable
decay_iters  = int(0.01 * max_iters)   # 1% cool down

# ── Distillation hyper-params ─────────────────────────────────────────────────
top_k = 100
top_p = 0.95

# ── Optimizer (LR set manually each step via WSD-LR) ─────────────────────────
max_lr    = 3e-4
min_lr    = 3e-5

# ── Output ────────────────────────────────────────────────────────────────────
out_dir       = 'distill/distill-gpt2-med'
eval_interval = 500
eval_iters    = 50
os.makedirs(out_dir, exist_ok=True)

## Load Teacher

Loads larger LLM and precomputes logits. Grabs sparse representations to avoid memory issues

In [10]:
# ── Teacher (GPT-2 medium, frozen) ────────────────────────────────────────────
teacher = GPT2LMHeadModel.from_pretrained('gpt2-medium', torch_dtype=dtype).to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print(f"Teacher: {sum(p.numel() for p in teacher.parameters())/1e6:.2f}M params")

TEACHER_V = 50257   # GPT-2 vocab; student uses 50304

Loading weights: 100%|██████████| 292/292 [00:05<00:00, 53.95it/s]


Teacher: 354.82M params


In [11]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 2         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4502 MiB |   8538 MiB |  99089 MiB |  94587 MiB |
|       from large pool |   4501 MiB |   8537 MiB |  98914 MiB |  94412 MiB |
|       from small pool |      1 MiB |      2 MiB |    175 MiB |    174 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   4502 MiB |   8538 MiB |  99089 MiB |  94587 MiB |
|       from large pool |   4501 MiB |   8537 MiB |  98914 MiB |

In [12]:
# Pre-compute teacher outputs for every training shard (skips completed shards)
for shard in train_shards:
    precompute_teacher_sparse(shard, teacher, top_k, top_p, block_size, batch_size)

# Free teacher GPU memory — not needed during student training
teacher.cpu()
torch.cuda.empty_cache()
print("Teacher unloaded from GPU")

# For loading shards once teacher probs and indices have been stored
def load_shard_distill(idx):
    path = train_shards[idx]
    return (
        np.load(path),
        np.load(path.replace('.npy', '_tidx.npy')),
        np.load(path.replace('.npy', '_tprob.npy')),
    )

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.30 GiB. GPU 0 has a total capacity of 10.57 GiB of which 476.00 MiB is free. Process 474446 has 10.10 GiB memory in use. Of the allocated memory 9.77 GiB is allocated by PyTorch, and 151.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Load student model

In [ ]:
# Load checkpoint from huggingface, if needed
from huggingface_hub import snapshot_download
model_dir = snapshot_download(
    repo_id="QUdie/UdieGPT", 
    local_dir="./model_chkpt"
)

Fetching 3 files: 100%|██████████| 3/3 [00:06<00:00,  2.01s/it]


In [ ]:
# ── Student (custom GPT loaded from checkpoint) ───────────────────────────────
ckpt_path  = f'{model_dir}/checkpoint.pt'
checkpoint = torch.load(ckpt_path, map_location=device)

checkpoint_model_args = checkpoint['model_args']
gptconf = GPTConfig(**checkpoint_model_args)
student = GPT(gptconf)

state_dict = checkpoint['model']
prefix = '_orig_mod.'
for k in list(state_dict):
    if k.startswith(prefix):
        state_dict[k[len(prefix):]] = state_dict.pop(k)
student.load_state_dict(state_dict)
student = student.to(device).to(dtype)
student.train()
print(f"Student: {student.get_num_params()/1e6:.2f}M params")

grad_clip = 1.0
optimizer = AdamW(student.parameters(), lr=max_lr, weight_decay=0.1, betas=(0.9, 0.95))

/tmp/ipykernel_2577/2303351313.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


number of parameters: 95.27M
Student: 95.27M params


Loading weights: 100%|██████████| 292/292 [00:05<00:00, 57.12it/s]


Teacher: 354.82M params


### Train student

In [ ]:
# ── Training loop for student ───────────────────────────────────────────────────────
log_steps, log_train, log_val, log_lam, log_lr = [], [], [], [], []
shard_idx, steps_on_shard = 0, 0
cur_data, cur_tidx, cur_tprob = load_shard_distill(shard_idx)

for step in range(max_iters):
    # Cycle to next shard once all chunks have been sampled
    if steps_on_shard * batch_size >= len(cur_data) // block_size:
        shard_idx += 1
        cur_data, cur_tidx, cur_tprob = load_shard_distill(shard_idx)
        steps_on_shard = 0
    
    student.train()

    # learning rate & KDL weighting
    lam = kd_lambda(step, warmup_iters, stable_iters, decay_iters)
    lr  = wsd_lr(lam, max_lr, min_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    x, y, ti, tp = get_train_batch(cur_data, cur_tidx, cur_tprob)
    steps_on_shard += 1

    with torch.amp.autocast(device_type=device_type, dtype=dtype):
        student_logits, ce = student(x, y)
        kd   = kd_loss_from_sparse(student_logits, ti, tp)
        loss = lam * kd + (1.0 - lam) * ce      # diff between teacher and diff between truth

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), grad_clip)
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f} | ce {ce.item():.4f} "
              f"| kd {kd.item():.4f} | α {lam:.3f} | lr {lr:.2e}")

    if step % eval_interval == 0:
        student.eval()
        v_losses = []
        with torch.no_grad():
            for _ in range(eval_iters):
                xv, yv = get_batch('val')
                with torch.amp.autocast(device_type=device_type, dtype=dtype):
                    _, vl = student(xv, yv)
                v_losses.append(vl.item())
        log_steps.append(step)
        log_train.append(loss.item())
        log_val.append(float(np.mean(v_losses)))
        log_lam.append(lam)
        log_lr.append(lr)
        print(f"  → val_loss {log_val[-1]:.4f}")

torch.save(
    {'model': student.state_dict(), 'model_args': vars(student.config)},
    os.path.join(out_dir, 'ckpt.pt'),
)
print(f"\nSaved distilled checkpoint → {out_dir}/ckpt.pt")

## Show results

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
axes[0].plot(log_steps, log_train, label='train loss')
axes[0].plot(log_steps, log_val,   label='val loss')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# α schedule
axes[1].plot(log_steps, log_lam, color='orange')
axes[1].axvline(warmup_iters,                label='warmup end',  color='gray',  linestyle='--')
axes[1].axvline(warmup_iters + stable_iters, label='decay start', color='brown', linestyle='--')
axes[1].set_xlabel('step'); axes[1].set_ylabel('α')
axes[1].set_title('WSD-α  (KD loss weight, cosine decay)')
axes[1].set_ylim(-0.05, 1.15); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# LR schedule
axes[2].plot(log_steps, log_lr, color='steelblue')
axes[2].axvline(warmup_iters,                label='warmup end',  color='gray',  linestyle='--')
axes[2].axvline(warmup_iters + stable_iters, label='decay start', color='brown', linestyle='--')
axes[2].set_xlabel('step'); axes[2].set_ylabel('learning rate')
axes[2].set_title('WSD-LR  (cosine decay)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(out_dir, 'distill_curves.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {save_path}")